In [11]:
import pandas as pd
import numpy as np

vehicle_df = pd.read_parquet(r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\vehicle_insurance\vehicle.parquet")
claims_df = pd.read_parquet(r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\analytics_vidhya_claims\combined_claims.parquet")

In [15]:
vehicle_df.head(5)

,id,gender,customer_age,has_driving_license,region,previously_insured,vehicle_age,vehicle_damage,Annual_Premium,sales_channel,days_since_last_contact,accepted_offer,_ingestion_timestamp,_source_file
0,1,Male,44,1,28.0,0,> 2 Years,True,40454.0,26.0,217,1,2026-07-30 13:37:36.721549+00:00,C:/Users/PC 12/.cache/kagglehub/datasets/apoor...
1,2,Male,76,1,3.0,0,1-2 Year,False,33536.0,26.0,183,0,2026-07-30 13:37:36.721549+00:00,C:/Users/PC 12/.cache/kagglehub/datasets/apoor...
2,3,Male,47,1,28.0,0,> 2 Years,True,38294.0,26.0,27,1,2026-07-30 13:37:36.721549+00:00,C:/Users/PC 12/.cache/kagglehub/datasets/apoor...
3,4,Male,21,1,11.0,1,< 1 Year,False,28619.0,152.0,203,0,2026-07-30 13:37:36.721549+00:00,C:/Users/PC 12/.cache/kagglehub/datasets/apoor...
4,5,Female,29,1,41.0,1,< 1 Year,False,27496.0,152.0,39,0,2026-07-30 13:37:36.721549+00:00,C:/Users/PC 12/.cache/kagglehub/datasets/apoor...


Customer Silver

In [16]:

customer_df = vehicle_df[
    [
        "gender",
        "customer_age",
        "has_driving_license",
        "previously_insured",
        "region"
    ]
]

Generate Keys

In [17]:
customer_df.insert(
    0,
    "customer_sk",
    range(1, len(customer_df) + 1)
)

Age Band

In [18]:
customer_df["age_band"] = pd.cut(
    customer_df["customer_age"],
    bins=[18,25,35,45,60,100],
    labels=["18-25","26-35","36-45","46-60","60+"]
)

Save

In [21]:
import os

file_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\customer.csv"

# Ensure the folder directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the file
customer_df.to_csv(file_path, index=False)

# Vehicle Silver

In [22]:
vehicle_df = claims_df.rename(
    columns={
        "age_of_car":"vehicle_age",
        "transmission_type":"transmission",
        "steering_type":"steering"
    }
)

vehicle_df = vehicle_df[
[
"policy_id",
"vehicle_age",
"make",
"model",
"fuel_type",
"segment",
"engine_type",
"displacement",
"cylinder",
"transmission",
"steering",
"length",
"width",
"height",
"gross_weight"
]
]

Generate Vehicle Key

In [23]:
vehicle_df.insert(
    0,
    "vehicle_sk",
    range(1, len(vehicle_df)+1)
)

Save

In [24]:
import os

file_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\vehicle.csv"

# Ensure the folder directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the file
customer_df.to_csv(file_path, index=False)

# Safety Table

In [25]:
safety_df = claims_df[
[
"policy_id",
"airbags",
"is_esc",
"is_tpms",
"is_parking_sensors",
"is_parking_camera",
"is_brake_assist",
"is_power_steering",
"is_speed_alert",
"ncap_rating"
]
]

Convert Yes/No

In [27]:
binary_columns = [
"is_esc",
"is_tpms",
"is_parking_sensors",
"is_parking_camera",
"is_brake_assist",
"is_power_steering",
"is_speed_alert"
]

for col in binary_columns:
    safety_df[col] = safety_df[col].map(
        {"Yes":1,"No":0}
    )

Safety Score

In [28]:
safety_df["safety_score"] = (
    safety_df["airbags"]
    + safety_df["is_esc"]
    + safety_df["is_tpms"]
    + safety_df["is_parking_sensors"]
    + safety_df["is_parking_camera"]
    + safety_df["is_brake_assist"]
    + safety_df["is_power_steering"]
    + safety_df["is_speed_alert"]
    + safety_df["ncap_rating"]
)

Save

In [29]:
import os

file_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\vehicle_safety.csv"

# Ensure the folder directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the file
customer_df.to_csv(file_path, index=False)